In [1]:
import os, gc, csv, json, math, random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

DTYPE = torch.bfloat16

MODEL_NAME = "Qwen/Qwen2.5-1.5B"

MAX_LENGTH = 2048
NUM_SHOT = 5
MMLU_EVAL_BATCH_SIZE = 4


FULL_9000_CHECKPOINT_ROOT = Path(
    "/kaggle/input/notebooks/tanmairaghava/"
    "phase1-lora-adapting/checkpoints"
)


PHASE4_ROOT = Path(
    "/kaggle/input/notebooks/tanmairaghava/"
    "less-phase4-training-random450-less450/"
    "less_phase4_training/experiments"
)

RANDOM450_CHECKPOINT_ROOT = (PHASE4_ROOT / "random450" / "checkpoints")
LESS450_CHECKPOINT_ROOT = (PHASE4_ROOT / "less450" / "checkpoints")


MMLU_ROOT = Path(
    "/kaggle/input/datasets/tanmairaghava/"
    "mmlu-data/mmlu"
)

MMLU_DEV_DIR = MMLU_ROOT / "dev"
MMLU_TEST_DIR = MMLU_ROOT / "test"

RESULTS_ROOT = Path("./less_phase5_mmlu_results")

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Device :", DEVICE)
print("Model  :", MODEL_NAME)
print("MMLU   :", MMLU_ROOT)

print("Full checkpoints :", FULL_9000_CHECKPOINT_ROOT)
print("Random-450 checkpoints :", RANDOM450_CHECKPOINT_ROOT)
print("LESS-450 checkpoints   :", LESS450_CHECKPOINT_ROOT)
print("Results :", RESULTS_ROOT)

Device : cuda
Model  : Qwen/Qwen2.5-1.5B
MMLU   : /kaggle/input/datasets/tanmairaghava/mmlu-data/mmlu
Full checkpoints : /kaggle/input/notebooks/tanmairaghava/phase1-lora-adapting/checkpoints
Random-450 checkpoints : /kaggle/input/notebooks/tanmairaghava/less-phase4-training-random450-less450/less_phase4_training/experiments/random450/checkpoints
LESS-450 checkpoints   : /kaggle/input/notebooks/tanmairaghava/less-phase4-training-random450-less450/less_phase4_training/experiments/less450/checkpoints
Results : less_phase5_mmlu_results


In [2]:
import sys
import subprocess

subprocess.check_call([sys.executable, "-m","pip","install","-q","--no-deps","torchao>=0.16.0"])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 19.7 MB/s eta 0:00:00


0

In [3]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer:", MODEL_NAME)
print("Vocab:", len(tokenizer))

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer: Qwen/Qwen2.5-1.5B
Vocab: 151665


# LESS — Phase 5
## 5-shot MMLU evaluation

This notebook performs **only evaluation**.

It evaluates four conditions:

1. Base Qwen2.5-1.5B
2. Full-9000 using the **existing Phase-1 LoRA checkpoints**
3. Random-450 checkpoints produced by the Phase-4 training notebook
4. LESS-450 checkpoints produced by the Phase-4 training notebook

There is **no training** in this notebook.

MMLU:
- 57 subjects
- first 5 `dev` examples are the fixed demonstrations
- all `test` examples remain held out
- final score = mean accuracy across the 57 subjects


In [4]:
dev_files = sorted(MMLU_DEV_DIR.glob("*_dev.csv"))
test_files = sorted(MMLU_TEST_DIR.glob("*_test.csv"))

dev_subjects = {
    p.name.removesuffix("_dev.csv")
    for p in dev_files
}
test_subjects = {
    p.name.removesuffix("_test.csv")
    for p in test_files
}

assert len(dev_subjects) == 57, f"Expected 57 dev subjects, found {len(dev_subjects)}"
assert len(test_subjects) == 57, f"Expected 57 test subjects, found {len(test_subjects)}"
assert dev_subjects == test_subjects, "Dev/test subject sets differ."

MMLU_SUBJECTS = sorted(dev_subjects)

def read_mmlu_csv(path: Path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)

        for row in reader:
            if not row:
                continue

            if len(row) < 6:
                raise ValueError(f"Malformed MMLU row in {path}: {row}")

            answer = row[5].strip()
            assert answer in ["A", "B", "C", "D"]

            rows.append({
                "question": row[0],
                "choices": row[1:5],
                "answer_letter": answer,
            })

    return rows


MMLU_DEV = {}
MMLU_TEST = {}

for subject in MMLU_SUBJECTS:
    dev_rows = read_mmlu_csv(
        MMLU_DEV_DIR / f"{subject}_dev.csv"
    )
    test_rows = read_mmlu_csv(
        MMLU_TEST_DIR / f"{subject}_test.csv"
    )

    assert len(dev_rows) >= NUM_SHOT

    MMLU_DEV[subject] = dev_rows
    MMLU_TEST[subject] = test_rows

MMLU_DEMOS = {
    subject: MMLU_DEV[subject][:NUM_SHOT]
    for subject in MMLU_SUBJECTS
}

print("Subjects :", len(MMLU_SUBJECTS))
print("5-shot demos:", sum(len(v) for v in MMLU_DEMOS.values()))

for subject in MMLU_SUBJECTS[:5]:
    print(subject, "->", len(MMLU_DEMOS[subject]), "demos,", len(MMLU_TEST[subject]), "test")


Subjects : 57
5-shot demos: 285
abstract_algebra -> 5 demos, 100 test
anatomy -> 5 demos, 135 test
astronomy -> 5 demos, 152 test
business_ethics -> 5 demos, 100 test
clinical_knowledge -> 5 demos, 265 test


In [5]:
ANSWER_LETTERS = ["A", "B", "C", "D"]

def mmlu_user_content(example):
    choices = example["choices"]

    return (
        f"{example['question']}\n\n"
        f"A. {choices[0]}\n"
        f"B. {choices[1]}\n"
        f"C. {choices[2]}\n"
        f"D. {choices[3]}"
    )

def build_few_shot_messages(demos, query):
    messages = []

    for demo in demos:
        messages.append({
            "role": "user",
            "content": mmlu_user_content(demo),
        })
        messages.append({
            "role": "assistant",
            "content": demo["answer_letter"],
        })

    messages.append({
        "role": "user",
        "content": mmlu_user_content(query),
    })

    return messages


# Verify the exact tokenizer IDs of A/B/C/D.
answer_token_ids = {}

for letter in ANSWER_LETTERS:
    ids = tokenizer.encode(
        letter,
        add_special_tokens=False,
    )

    if len(ids) != 1:
        raise RuntimeError(
            f"MMLU answer {letter!r} is not a single token for "
            f"{MODEL_NAME}: ids={ids}"
        )

    answer_token_ids[letter] = ids[0]

print("Answer token IDs:", answer_token_ids)


Answer token IDs: {'A': 32, 'B': 33, 'C': 34, 'D': 35}


In [6]:
def load_eval_base():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        dtype=DTYPE,
        trust_remote_code=True,
    )

    model.config.use_cache = True
    model.to(DEVICE)
    model.eval()

    return model


def load_eval_checkpoint(checkpoint_dir):
    base_model = load_eval_base()

    model = PeftModel.from_pretrained(
        base_model,
        checkpoint_dir,
        is_trainable=False,
    )

    model.eval()
    return model


def cleanup_eval_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


In [7]:
@torch.inference_mode()
def evaluate_mmlu_model(
    model,
    model_name,
    batch_size=MMLU_EVAL_BATCH_SIZE,
):
    model.eval()

    subject_results = []

    for subject in tqdm(
        MMLU_SUBJECTS,
        desc=f"MMLU | {model_name}",
    ):
        demos = MMLU_DEMOS[subject]
        test_rows = MMLU_TEST[subject]

        correct = 0
        total = len(test_rows)

        prompts = [
            tokenizer.apply_chat_template(
                build_few_shot_messages(demos, row),
                tokenize=False,
                add_generation_prompt=True,
            )
            for row in test_rows
        ]

        gold = [row["answer_letter"] for row in test_rows]

        for start in range(0, total, batch_size):
            batch_prompts = prompts[start:start + batch_size]

            encoded = tokenizer(
                batch_prompts,
                padding=True,
                truncation=True,
                max_length=MAX_LENGTH,
                return_tensors="pt",
                add_special_tokens=False,
            )

            input_ids = encoded["input_ids"].to(
                DEVICE,
                non_blocking=True,
            )
            attention_mask = encoded["attention_mask"].to(
                DEVICE,
                non_blocking=True,
            )

            with torch.autocast(
                device_type="cuda",
                dtype=DTYPE,
            ):
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    use_cache=True,
                    return_dict=True,
                )

            # Right-padding: gather the logits at the final actual token.
            last_positions = attention_mask.sum(dim=1) - 1
            row_ids = torch.arange(
                input_ids.shape[0],
                device=DEVICE,
            )

            next_token_logits = outputs.logits[
                row_ids,
                last_positions,
            ]

            candidate_logits = torch.stack([
                next_token_logits[:, answer_token_ids["A"]],
                next_token_logits[:, answer_token_ids["B"]],
                next_token_logits[:, answer_token_ids["C"]],
                next_token_logits[:, answer_token_ids["D"]],
            ], dim=1)

            predictions = candidate_logits.argmax(
                dim=1
            ).detach().cpu().tolist()

            for pred_idx, pred in enumerate(predictions):
                predicted_letter = ANSWER_LETTERS[pred]
                if predicted_letter == gold[start + pred_idx]:
                    correct += 1

            del encoded, input_ids, attention_mask
            del outputs, last_positions, row_ids
            del next_token_logits, candidate_logits

        accuracy = correct / total

        subject_results.append({
            "subject": subject,
            "correct": correct,
            "total": total,
            "accuracy": accuracy,
        })

    mmlu_score = float(
        np.mean([
            x["accuracy"]
            for x in subject_results
        ])
    )

    result = {
        "model": model_name,
        "mmlu_5shot": mmlu_score,
        "num_subjects": len(subject_results),
        "subjects": subject_results,
    }

    print()
    print("=" * 80)
    print(f"MMLU RESULT: {model_name}")
    print("=" * 80)
    print(f"5-shot MMLU: {100 * mmlu_score:.4f}%")

    return result

## Restartable evaluation

The Base model has already been evaluated:

**Base MMLU 5-shot = 46.72055329996985%**

It is intentionally skipped.

For Full-9000, Random-450, and LESS-450, every completed epoch is saved
immediately as a JSON file. If a Kaggle session stops, rerun the evaluation
cell; existing epoch results are detected and skipped automatically.

In [ ]:
all_eval_results = []

PREVIOUS_RESULTS_ROOT = Path(
    "/kaggle/input/datasets/lokeswarareddyp/epoch1and2/"
    "less_phase5_mmlu_results"
)

print("=" * 80)
print("PREVIOUS RESULTS")
print("=" * 80)
print("Source:", PREVIOUS_RESULTS_ROOT)


def result_path_for(experiment_name, epoch):
    """
    Return the output path for the current run.
    """
    return RESULTS_ROOT / (
        f"mmlu_{experiment_name}_epoch{epoch}.json"
    )


def previous_result_path_for(experiment_name, epoch):
    """
    Return the path of a result from the uploaded previous-run dataset.
    """
    return PREVIOUS_RESULTS_ROOT / (
        f"mmlu_{experiment_name}_epoch{epoch}.json"
    )


def load_result(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


for epoch in [1, 2]:

    previous_path = previous_result_path_for(
        "random450",
        epoch
    )

    if not previous_path.exists():
        raise FileNotFoundError(
            f"Expected previous result not found:\n"
            f"{previous_path}"
        )

    result = load_result(previous_path)

    all_eval_results.append(result)

    print(
        f"Recovered Random-450 epoch {epoch}: "
        f"{100.0 * result['mmlu_5shot']:.4f}%"
    )


for epoch in [3, 4]:

    result_path = result_path_for(
        "random450",
        epoch
    )

    # If this notebook/output directory already contains it,
    # don't recompute it.
    if result_path.exists():

        print()
        print("=" * 80)
        print(f"SKIPPING Random-450 | EPOCH {epoch}")
        print(f"Existing result: {result_path}")
        print("=" * 80)

        result = load_result(result_path)

        all_eval_results.append(result)

        print(
            f"Recovered MMLU: "
            f"{100.0 * result['mmlu_5shot']:.4f}%"
        )

        continue


    checkpoint_dir = (
        RANDOM450_CHECKPOINT_ROOT /
        f"epoch_{epoch}"
    )

    if not checkpoint_dir.exists():
        raise FileNotFoundError(
            f"Missing Radom-450 checkpoint:\n"
            f"{checkpoint_dir}"
        )


    print()
    print("=" * 80)
    print(f"EVALUATING Radom-450 | EPOCH {epoch}")
    print("=" * 80)
    print(f"Checkpoint: {checkpoint_dir}")


    model = None

    try:

        model = load_eval_checkpoint(
            checkpoint_dir
        )

        result = evaluate_mmlu_model(
            model,
            f"random450_epoch{epoch}"
        )

        result["experiment"] = "random450"
        result["epoch"] = epoch
        result["checkpoint"] = str(
            checkpoint_dir
        )

        with open(
            result_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                result,
                f,
                indent=2
            )


        all_eval_results.append(result)

        print()
        print(
            f"Saved: {result_path}"
        )

        print(
            f"Random-450 epoch {epoch}: "
            f"{100.0 * result['mmlu_5shot']:.4f}%"
        )


    finally:

        if model is not None:
            cleanup_eval_model(model)


print()
print("=" * 80)
print("Random-450 EVALUATION STATUS")
print("=" * 80)

for result in sorted(
    all_eval_results,
    key=lambda x: x.get("epoch", 0)
):

    print(
        f"Radom-450 epoch "
        f"{result.get('epoch', '?')}: "
        f"{100.0 * result['mmlu_5shot']:.4f}%"
    )

PREVIOUS RESULTS
Source: /kaggle/input/datasets/lokeswarareddyp/epoch1and2/less_phase5_mmlu_results
Recovered Random-450 epoch 1: 44.8556%
Recovered Random-450 epoch 2: 37.1535%

EVALUATING Radom-450 | EPOCH 3
Checkpoint: /kaggle/input/notebooks/tanmairaghava/less-phase4-training-random450-less450/less_phase4_training/experiments/random450/checkpoints/epoch_3


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


...


MMLU RESULT: random450_epoch3
5-shot MMLU: 34.8504%

Saved: less_phase5_mmlu_results/mmlu_random450_epoch3.json
Random-450 epoch 3: 34.8504%

EVALUATING Radom-450 | EPOCH 4
Checkpoint: /kaggle/input/notebooks/tanmairaghava/less-phase4-training-random450-less450/less_phase4_training/experiments/random450/checkpoints/epoch_4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

MMLU | random450_epoch4:   0%|          | 0/57 [00:00<?, ?it/s]


MMLU RESULT: random450_epoch4
5-shot MMLU: 33.9075%

Saved: less_phase5_mmlu_results/mmlu_random450_epoch4.json
Random-450 epoch 4: 33.9075%

Random-450 EVALUATION STATUS
Radom-450 epoch 1: 44.8556%
Radom-450 epoch 2: 37.1535%
Radom-450 epoch 3: 34.8504%
Radom-450 epoch 4: 33.9075%


NameError: name 'kaggle' is not defined